In [ ]:
import io, warnings, numpy as np, pandas as pd
from sklearn.linear_model import Ridge
from sklearn.kernel_ridge import KernelRidge
warnings.filterwarnings("ignore")

CSV = """
query,container_count,declared_value,discrepancy_ratio,port,prior_shipments,recent_seizures,route_age_days,shipper_score,shipper_years,transfers,score
R1-1,50,50,0.5,A,10,2.5,46.5,600,20,3,0.3544
R1-2,50,50,0,A,10,2.5,46.5,600,20,3,0.1797
R1-3,50,100,1,A,10,2.5,46.5,600,20,3,0.6285
R1-4,50,100,0.5,A,10,2.5,46.5,600,20,3,0.3546
R1-5,50,100,1,A,10,2.5,46.5,900,20,3,0.7139
R1-6,50,100,0.5,A,10,2.5,46.5,900,20,3,0.4891
R1-7,50,100,1,A,20,2.5,46.5,900,20,3,0.7139
R1-8,50,100,1,A,20,0,46.5,900,20,3,0.6430
R1-9,50,100,1,A,20,5,46.5,900,20,3,0.8265
R1-10,50,100,1,A,20,5,75,900,20,3,0.6238
R1-11,50,100,1,A,20,5,46.5,900,40,3,0.8150
R1-12,50,100,1,A,20,5,46.5,900,20,6,0.8265
R1-13,50,100,1,A,20,5,46.5,900,0,6,0.7819
R1-14,50,100,1,A,20,5,46.5,900,20,6,0.8265
R1-15,100,100,1,A,20,5,46.5,900,20,6,0.8265
R1-16,100,100,1,B,20,5,46.5,900,20,6,0.8327
R1-17,100,100,1,B,20,5,18,900,20,6,0.8858
R1-18,100,100,1,C,20,5,18,900,20,6,0.0430
R1-19,100,100,1,B,20,5,18,900,20,6,0.8858
R1-20,100,100,1,D,20,5,18,900,20,6,0.8879
R1-21,100,100,1,D,20,5,18,609,20,6,0.8440
R1-22,100,100,1,D,20,5,18,900,20,6,0.8879
R1-23,100,100,1,D,20,5,18,900,0,6,0.8595
R1-24,100,100,1,D,20,5,18,900,40,6,0.8772
R1-25,100,100,1,D,20,5,18,900,20,6,0.8879
R1-26,100,100,1,D,20,5,18,900,16.6,6,0.8826
R1-27,100,100,1,D,20,5,18,900,32.4,6,0.8972
R1-28,100,100,1,D,20,5,18,900,36.6,6,0.8834
R1-29,100,100,1,D,20,5,18,900,30,6,0.8955
R1-30,100,100,1,D,20,5,18,900,32,6,0.8969
R1-31,100,100,1,D,20,5,18,900,32.5,6,0.8972
R1-32,100,100,1,D,20,5,18,816,32.5,6,0.8941
R1-33,100,100,1,D,20,5,18,900,32.5,6,0.8972
R1-34,100,71,1,D,20,5,18,900,32.5,6,0.9430
R1-35,100,49.5,1,D,20,5,18,900,32.5,6,0.9459
R1-36,100,65.5,1,D,20,5,18,900,32.5,6,0.9397
R1-37,100,70,1,D,20,5,18,900,32.5,6,0.9430
R1-38,78.5,70,1,D,20,5,18,900,32.5,6,0.9430
R2-1,50,50,0.5,A,10,2.5,46.5,600,20,3,0.3544
R2-2,0,50,0.5,A,10,2.5,46.5,600,20,3,0.3544
R2-3,100,50,0.5,A,10,2.5,46.5,600,20,3,0.3544
R2-4,50,0,0.5,A,10,2.5,46.5,600,20,3,0.5488
R2-5,50,100,0.5,A,10,2.5,46.5,600,20,3,0.3546
"""
df = pd.read_csv(io.StringIO(CSV.strip()))
print(df.shape)

(43, 12)


In [ ]:
RANGES = {"container_count":(0,100),"declared_value":(0,100),"discrepancy_ratio":(0,1),"prior_shipments":(0,20),
          "recent_seizures":(0,5),"route_age_days":(18,75),"shipper_score":(300,900),"shipper_years":(0,40),"transfers":(0,6)}
PORTS = list("ABCD"); NUM = [k for k in RANGES if k != "container_count"]
logit = lambda p: np.log(p/(1-p)); sigm = lambda z: 1/(1+np.exp(-z))

U = df.drop_duplicates(subset=NUM+["port"]).reset_index(drop=True)   # 32 distinct inputs
def numeric(d): return np.column_stack([(d[k]-RANGES[k][0])/(RANGES[k][1]-RANGES[k][0]) for k in NUM])
def onehot(d):  return np.eye(4)[d["port"].map({p:i for i,p in enumerate(PORTS)}).values]*1.7
X = np.hstack([numeric(U), onehot(U)]); score = U.score.values; y = logit(np.clip(score,1e-4,1-1e-4))
print("distinct inputs:", len(U))

distinct inputs: 32


In [ ]:
def held_out(make):
    pred = np.zeros(len(y))
    for i in range(len(y)):
        tr = np.arange(len(y)) != i; mu = y[tr].mean()
        pred[i] = sigm(make().fit(X[tr], y[tr]-mu).predict(X[[i]])[0] + mu)
    return pred

models = {"Ridge (linear)": lambda: Ridge(alpha=1e-3),
          "Kernel Ridge (RBF)": lambda: KernelRidge(kernel="rbf", gamma=1/1.44, alpha=1e-3)}
res = {}
for n, mk in models.items():
    p = held_out(mk); ok = (p >= .5) == (score >= .5); res[n] = p
    print(f"{n:20s} held-out decision accuracy = {ok.mean():.2%} ({ok.sum()}/{len(ok)})  misses: {list(U['query'][~ok])}")
best = max(res, key=lambda n: ((res[n]>=.5)==(score>=.5)).mean())
print("best:", best)

Ridge (linear)       held-out decision accuracy = 96.88% (31/32)  misses: ['R1-18']
Kernel Ridge (RBF)   held-out decision accuracy = 93.75% (30/32)  misses: ['R1-18', 'R2-4']
best: Ridge (linear)


In [ ]:
final = Ridge(alpha=1e-3); MU = y.mean(); final.fit(X, y-MU)

def predict(**inp):
    for k,(lo,hi) in RANGES.items():
        if not lo <= inp[k] <= hi: raise ValueError(f"{k} must be in [{lo}, {hi}]")
    q = pd.DataFrame([inp])
    hit = df[(df[NUM] == q[NUM].iloc[0]).all(axis=1) & (df.port == inp["port"])]
    s = float(hit.score.iloc[0]) if len(hit) else float(sigm(final.predict(np.hstack([numeric(q), onehot(q)]))[0] + MU))
    return round(s,4), ("APPROVE" if s >= .5 else "DECLINE")

cols = list(RANGES)+["port"]
known = np.mean([predict(**r[cols].to_dict())[1] == ("APPROVE" if r.score>=.5 else "DECLINE") for _,r in df.iterrows()])
print(f"known-query accuracy (the 43 asked queries): {known:.2%}  <- memorised, not a generalisation score")
print(f"held-out accuracy (new inputs):              {((res[best]>=.5)==(score>=.5)).mean():.2%}  <- the honest number")

known-query accuracy (the 43 asked queries): 100.00%  <- memorised, not a generalisation score
held-out accuracy (new inputs):              96.88%  <- the honest number


In [ ]:
base = dict(container_count=50, declared_value=50, discrepancy_ratio=.5, port="A", prior_shipments=10,
            recent_seizures=2.5, route_age_days=46.5, shipper_score=600, shipper_years=20, transfers=3)
print(predict(**base))
print(predict(**{**base, "discrepancy_ratio": .8, "shipper_score": 800}))

(0.3544, 'DECLINE')
(0.7013, 'APPROVE')


In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, confusion_matrix

actual_dec = score >= 0.5                       # True = APPROVE
rows = []
for name, p in res.items():                     # p = held-out predicted scores
    pred_dec = p >= 0.5
    tn, fp, fn, tp = confusion_matrix(actual_dec, pred_dec, labels=[False, True]).ravel()
    mse = mean_squared_error(score, p)
    rows.append({"Model": name,
                 "Decision accuracy %": 100*(tp+tn)/len(score),
                 "R2": r2_score(score, p),
                 "MSE": mse, "RMSE": mse**0.5,
                 "MAE": mean_absolute_error(score, p),
                 "Worst miss": np.abs(p-score).max(),
                 "Correct": f"{tp+tn}/{len(score)}",
                 "DECLINE->APPROVE (wrong)": fp, "APPROVE->DECLINE (wrong)": fn})
report = pd.DataFrame(rows).set_index("Model").round(4)
pd.set_option("display.width", 200)
print("HELD-OUT RESULTS (each point was hidden, then predicted)\n")
print(report.T)

b = report.loc[best]
print(f"""
===== PLAIN-ENGLISH SUMMARY for {best} =====
Decision accuracy : {b['Decision accuracy %']:.2f}%  -> got {b['Correct']} APPROVE/DECLINE decisions right on unseen inputs
R2                : {b['R2']:.3f}   -> explains {b['R2']*100:.1f}% of the variation in scores (1.0 = perfect, 0 = no better than guessing the average)
MSE               : {b['MSE']:.4f}  -> average of (error squared); big misses count extra
RMSE              : {b['RMSE']:.4f}  -> typical error size in score units (same scale as the score)
MAE               : {b['MAE']:.4f}  -> average error is about {b['MAE']:.3f} score points
Worst miss        : {b['Worst miss']:.4f}  -> the single biggest error (this is the port C point)
""")

# the same metrics without the one port-C point, to see how the model does everywhere else
m = U["query"] != "R1-18"
for n, p in res.items():
    print(f"{n}: R2 without port-C point = {r2_score(score[m], p[m]):.3f}, MAE = {mean_absolute_error(score[m], p[m]):.3f}")

HELD-OUT RESULTS (each point was hidden, then predicted)

Model                    Ridge (linear) Kernel Ridge (RBF)
Decision accuracy %              96.875              93.75
R2                               0.5316             0.6008
MSE                              0.0246             0.0209
RMSE                             0.1568             0.1447
MAE                              0.0566              0.051
Worst miss                       0.8544             0.7677
Correct                           31/32              30/32
DECLINE->APPROVE (wrong)              1                  1
APPROVE->DECLINE (wrong)              0                  1

===== PLAIN-ENGLISH SUMMARY for Ridge (linear) =====
Decision accuracy : 96.88%  -> got 31/32 APPROVE/DECLINE decisions right on unseen inputs
R2                : 0.532   -> explains 53.2% of the variation in scores (1.0 = perfect, 0 = no better than guessing the average)
MSE               : 0.0246  -> average of (error squared); big misses count ex